#  Livrable 1 – Classification binaire d’images

##  Objectif
Développer un modèle de **classification binaire** capable de distinguer :
- **Photos**
- **Autres images** (schémas, textes scannés, peintures…)

Ce livrable constitue la **première étape** du workflow demandé par TouNum.

In [ ]:
# Import des librairies
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [1]:
from platformdirs import macos
import sys
import keras
import pandas as pd
import sklearn as sk
import scipy as sp
import tensorflow as tf
import platform

print (f"Python Platform: {platform.platform ()}")
print (f"Tensor Flow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print ()

print (f"Python {sys.version}")
print (f"Pandas {pd.__version__}")
print (f"Scikit-Learn {sk.__version__}")
print (f"SciPy {sp.__version__}")
gpu = len (tf.config.list_physical_devices ('GPU'))>0
print ("GPU is", "available" if gpu else "NOT AVAILABLE")

Python Platform: macOS-26.0.1-arm64-arm-64bit
Tensor Flow Version: 2.16.2
Keras Version: 3.11.3

Python 3.10.8 (v3.10.8:aaaf517424, Oct 11 2022, 10:14:40) [Clang 13.0.0 (clang-1300.0.29.30)]
Pandas 2.3.3
Scikit-Learn 1.7.2
SciPy 1.15.3
GPU is available


In [3]:
import sys
import os
import platform

# Python info
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())

# Check if running inside a virtual environment
print("In virtualenv:", hasattr(sys, 'real_prefix') or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix))
print("Virtualenv path:", sys.prefix)

# Check installed packages
try:
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
    print("Available GPUs:", tf.config.list_physical_devices('GPU'))
except Exception as e:
    print("TensorFlow import error:", e)

try:
    import keras
    print("Keras version:", keras.__version__)
except Exception as e:
    print("Keras import error:", e)


Python executable: /Users/lukashouille/venv-metal/bin/python
Python version: 3.10.8 (v3.10.8:aaaf517424, Oct 11 2022, 10:14:40) [Clang 13.0.0 (clang-1300.0.29.30)]
Platform: macOS-26.0.1-arm64-arm-64bit
In virtualenv: True
Virtualenv path: /Users/lukashouille/venv-metal
TensorFlow version: 2.16.2
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Keras version: 3.11.3


In [ ]:
import tensorflow as tf

cifar = tf.keras.datasets.cifar100
(x_train, y_train), (x_test, y_test) = cifar.load_data()

model = tf.keras.applications.ResNet50(
    include_top=True,
    weights=None,
    input_shape=(32, 32, 3),
    classes=100,
)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
model.compile(optimizer="adam", loss=loss_fn, metrics=["accuracy"])
model.fit(x_train, y_train, epochs=5, batch_size=64)


In [ ]:
# Définition des chemins
data_dir = '../data/raw/'

In [ ]:
# Chargement des données
# Supposons que les images sont organisées en sous-dossiers 'photos' et 'autres'
# Supposons que les images dans autres sont organisées en sous dossier qu'il faut ignorer
photos_dir = os.path.join(data_dir, 'photos')
autres_dir = os.path.join(data_dir, 'non-photos')

In [ ]:
# Préparation des données
img_height, img_width = 128, 128
batch_size = 32

In [ ]:
# Générateur de données avec augmentation
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

In [ ]:
# Chargement des images
train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

In [ ]:
# Générateur de validation
validation_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

In [ ]:
# Construction du modèle CNN
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

In [ ]:
# Compilation du modèle
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Affichage du résumé du modèle
model.summary()

In [ ]:
# Diagramme du modèle
tf.keras.utils.plot_model(model, to_file='model_structure.png', show_shapes=True)

In [ ]:
# Entraînement du modèle avec Early Stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size,
    epochs=30,
    callbacks=[early_stopping]
)

In [ ]:
# Évaluation du modèle
loss, accuracy = model.evaluate(validation_generator)
print(f'Validation Loss: {loss}')
print(f'Validation Accuracy: {accuracy}')

In [ ]:
# Prédictions sur l'ensemble de validation
validation_generator.reset()
Y_pred = model.predict(validation_generator)
y_pred = (Y_pred > 0.5).astype(int)

In [ ]:
# Rapport de classification
print('Classification Report')
print(classification_report(validation_generator.classes, y_pred, target_names=['Autres', 'Photos']))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(validation_generator.classes, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Autres', 'Photos'], yticklabels=['Autres', 'Photos'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Courbes d'apprentissage
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')

In [ ]:
# Courbe de perte
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# Sauvegarde du modèle
model_version = 1
while os.path.exists(f'model_v{model_version}.h5'):
    model_version += 1
model.save(f'model_v{model_version}.h5')

In [ ]:
# Fin du livrable 1